Stream-Moore: Incremental Modularity-based Community Detection

Algorithm overview

1. Initialise each node in its own community.
2. Stream edges one by one (or in batches).
3. For each new edge (u, v), evaluate the modularity gain (ΔQ) of merging
   the communities of u and v.
4. If ΔQ > 0, merge the two communities.
5. Track community assignments and modularity over time.

Modularity (Newman-Girvan):
    Q = (1/2m) * Σ_ij [ A_ij - k_i*k_j / (2m) ] * δ(c_i, c_j)

Modularity gain from merging communities C_a and C_b:
    ΔQ = [ e_ab / m - (a_a * a_b) / (2m)^2 ] * 2
where:
    e_ab  = number of edges between C_a and C_b (normalised by m)
    a_a   = sum of degrees of nodes in C_a
    a_b   = sum of degrees of nodes in C_b
    m     = total number of edges

In [3]:
import numpy as np
import scipy.sparse as sp
from collections import defaultdict
import time
import os

class UnionFind:

    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n
        self.size = [1] * n          # number of nodes in community
        self.a = None                # degree sums — set after degrees known

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]   # path compression
            x = self.parent[x]
        return x

    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            return False                                    # already same community
        # union by rank
        if self.rank[rx] < self.rank[ry]:
            rx, ry = ry, rx
        self.parent[ry] = rx
        self.size[rx] += self.size[ry]
        if self.a is not None:
            self.a[rx] += self.a[ry]
        if self.rank[rx] == self.rank[ry]:
            self.rank[rx] += 1
        return True


class IncrementalModularity:


    def __init__(self, degrees, m):

        self.degrees = np.array(degrees, dtype=np.float64)
        self.m = float(m)
        self.two_m = 2.0 * m

        # e[root_a][root_b] = edge count between community a and b
        # stored as nested defaultdict for sparsity
        self.e = defaultdict(lambda: defaultdict(float))

        # a[root] = sum of degrees of nodes in community root
        self._a = self.degrees.copy()   # initially each node is its own community

    def a(self, root):
        return self._a[root]

    def update_a(self, root, delta):
        self._a[root] += delta

    def merge_a(self, root_keep, root_remove):
        self._a[root_keep] += self._a[root_remove]
        self._a[root_remove] = 0.0


    def delta_q(self, ca, cb, extra_edge_weight=0.0):
 
        e_ab = self.e[ca][cb] + self.e[cb][ca] + extra_edge_weight
        a_ca = self._a[ca]
        a_cb = self._a[cb]
        dq = (e_ab / self.m) - (a_ca * a_cb) / (self.two_m ** 2)
        return 2.0 * dq


    def add_edge(self, ca, cb, weight=1.0):
        if ca == cb:
            return
        self.e[ca][cb] += weight
        self.e[cb][ca] += weight


    def merge_communities(self, ca, cb):

        # Absorb all edges of cb into ca
        for other, count in list(self.e[cb].items()):
            if other == ca:
                # edges between ca and cb become internal — drop them
                del self.e[ca][cb]
                del self.e[cb][ca]
                continue
            self.e[ca][other] = self.e[ca].get(other, 0.0) + count
            self.e[other][ca] = self.e[other].get(ca, 0.0) + count
            # remove old cb references
            if cb in self.e[other]:
                del self.e[other][cb]

        # clean up cb row
        if cb in self.e:
            del self.e[cb]

        # merge degree sums
        self.merge_a(ca, cb)

        return ca

    def modularity(self, uf):

        # collect internal edge counts per community
        roots = set(uf.find(i) for i in range(len(self.degrees)))
        Q = 0.0
        for r in roots:
            e_cc = self.e[r].get(r, 0.0)  # internal edges (self-loop in e means intra)
            a_r = self._a[r]
            Q += (e_cc / self.m) - (a_r / self.two_m) ** 2
        return Q


def stream_moore(adj: sp.spmatrix,
                 batch_size: int = 10_000,
                 min_dq: float = 0.0,
                 verbose: bool = True):

    adj = adj.tocsr().astype(np.float64)
    n = adj.shape[0]

    # Compute degrees
    degrees = np.array(adj.sum(axis=1)).flatten()
    m = adj.nnz / 2          # undirected: each edge stored twice
    two_m = 2.0 * m

    if verbose:
        print(f"Graph: {n:,} nodes | {int(m):,} edges | mean degree {degrees.mean():.2f}")

    # ---- Initialise ----
    uf = UnionFind(n)
    uf.a = None   # we use inc_mod._a instead

    inc = IncrementalModularity(degrees, m)

    # Stream edges (upper triangle to avoid duplicates)
    adj_coo = sp.triu(adj, k=1).tocoo()
    rows = adj_coo.row
    cols = adj_coo.col
    weights = adj_coo.data

    n_edges = len(rows)
    merges = 0
    history = []

    t0 = time.time()

    for idx in range(n_edges):
        u, v, w = int(rows[idx]), int(cols[idx]), float(weights[idx])

        cu = uf.find(u)
        cv = uf.find(v)

        if cu != cv:
            # Pass current edge weight: e[cu][cv] is 0 if this is the first
            # edge between these communities, so we must include it explicitly
            dq = inc.delta_q(cu, cv, extra_edge_weight=w)
            if dq > 0:
                # Merge smaller into larger (by degree sum for stability)
                if inc.a(cu) < inc.a(cv):
                    cu, cv = cv, cu
                uf.union(cu, cv)
                inc.merge_communities(cu, cv)
                merges += 1
            else:
                # Register the edge between communities without merging
                inc.add_edge(cu, cv, w)
        # If same community already, edge is internal — no action needed

        # Progress reporting
        if verbose and (idx + 1) % batch_size == 0:
            n_comm = len(set(uf.find(i) for i in range(n)))
            Q = inc.modularity(uf)
            elapsed = time.time() - t0
            history.append((idx + 1, Q, n_comm))
            print(f"  Edges {idx+1:>8,}/{n_edges:,} | "
                  f"Communities: {n_comm:>6,} | "
                  f"Merges: {merges:>6,} | "
                  f"Q ≈ {Q:.4f} | "
                  f"Elapsed: {elapsed:.1f}s")

    # Final state
    n_comm_final = len(set(uf.find(i) for i in range(n)))
    Q_final = inc.modularity(uf)
    elapsed = time.time() - t0

    if verbose:
        print(f"\n{'='*60}")
        print(f"Stream-Moore complete in {elapsed:.1f}s")
        print(f"  Final communities : {n_comm_final:,}")
        print(f"  Total merges      : {merges:,}")
        print(f"  Final modularity Q: {Q_final:.6f}")
        print(f"{'='*60}")

    # Build label array
    labels = np.array([uf.find(i) for i in range(n)])

    # Remap labels to 0-indexed integers
    unique_roots = {r: i for i, r in enumerate(sorted(set(labels)))}
    labels = np.array([unique_roots[l] for l in labels])

    history.append((n_edges, Q_final, n_comm_final))
    return labels, Q_final, history


def community_stats(labels):

    unique, counts = np.unique(labels, return_counts=True)
    n_comm = len(unique)
    print(f"\nCommunity statistics")
    print(f"  Number of communities : {n_comm:,}")
    print(f"  Largest community     : {counts.max():,} nodes")
    print(f"  Smallest community    : {counts.min():,} nodes")
    print(f"  Median size           : {np.median(counts):.0f} nodes")
    print(f"  Mean size             : {counts.mean():.1f} nodes")

    # Size distribution buckets
    buckets = [1, 2, 5, 10, 50, 100, 500, 1000, np.inf]
    print(f"\n  Size distribution:")
    for lo, hi in zip(buckets[:-1], buckets[1:]):
        mask = (counts >= lo) & (counts < hi)
        label = f"  [{int(lo):>5}, {int(hi):>5})" if hi != np.inf else f"  [{int(lo):>5},   inf)"
        print(f"    {label} : {mask.sum():,} communities")

    return unique, counts


if __name__ == "__main__":
    import sys

    # Define your default path here
    default_path = '/Users/antoniaspoerk/Desktop/Digital Neuroscience /Social Media Analytics/epilepsy_pediatrics_EEG/data/graphs/adjacency_sparse/inter_to_ict_chb01_03_2980_3010_adjacency_sparse.npz'

    # Check if we are in Jupyter
    is_jupyter = any("-f" in arg for arg in sys.argv)

    # If we have arguments and NOT in Jupyter, use the argument
    if len(sys.argv) > 1 and not is_jupyter:
        path = sys.argv[1]
    else:
        # Otherwise (in Jupyter or no args), use the default path
        path = default_path

    print(f"Loading graph from: {path}")
    
    # This should now print the actual path, not '-f'
    adj = sp.load_npz(path)

    # Run Stream-Moore
    labels, Q, history = stream_moore(
        adj,
        batch_size=200_000,
        min_dq=0.0,
        verbose=True
    )

    # Community statistics
    community_stats(labels)

    # Save results
    output_dir = "/Users/antoniaspoerk/Desktop/Digital Neuroscience /Social Media Analytics/epilepsy_pediatrics_EEG/src/03_analytics/outputs"
    os.makedirs(output_dir, exist_ok=True)

    import csv
    # Save results
    out_labels = os.path.join(output_dir, "community_labels.npy")
    np.save(out_labels, labels)
    print(f"\nCommunity labels saved to: {out_labels}")

    # Save history as CSV
    import csv
    out_history = os.path.join(output_dir, "modularity_history.csv")
    
    try:
        with open(out_history, "w", newline="", encoding="utf-8") as f:
            # TRY SEMICOLON (;) - This often fixes the "shit" look in Excel
            writer = csv.writer(f, delimiter=';') 
            
            # Write the header
            writer.writerow(["edges_processed", "modularity_Q", "n_communities"])
            
            # Write the data rows
            writer.writerows(history)
            
        print(f" Modularity history saved to: {out_history}")
    except Exception as e:
        print(f"Failed to save CSV: {e}")

Loading graph from: /Users/antoniaspoerk/Desktop/Digital Neuroscience /Social Media Analytics/epilepsy_pediatrics_EEG/data/graphs/adjacency_sparse/inter_to_ict_chb01_03_2980_3010_adjacency_sparse.npz
Graph: 176,640 nodes | 1,978,201 edges | mean degree 22.00
  Edges  200,000/1,978,201 | Communities:  7,665 | Merges: 168,975 | Q ≈ -0.0001 | Elapsed: 0.5s
  Edges  400,000/1,978,201 | Communities:  7,665 | Merges: 168,975 | Q ≈ -0.0001 | Elapsed: 0.6s
  Edges  600,000/1,978,201 | Communities:  7,665 | Merges: 168,975 | Q ≈ -0.0001 | Elapsed: 0.8s
  Edges  800,000/1,978,201 | Communities:  7,665 | Merges: 168,975 | Q ≈ -0.0001 | Elapsed: 0.9s
  Edges 1,000,000/1,978,201 | Communities:  7,665 | Merges: 168,975 | Q ≈ -0.0001 | Elapsed: 1.1s
  Edges 1,200,000/1,978,201 | Communities:  7,665 | Merges: 168,975 | Q ≈ -0.0001 | Elapsed: 1.2s
  Edges 1,400,000/1,978,201 | Communities:  7,665 | Merges: 168,975 | Q ≈ -0.0001 | Elapsed: 1.4s
  Edges 1,600,000/1,978,201 | Communities:  7,665 | Merges: